# 04 — One-stage anchor-free detector: YOLOv8 (Ultralytics)

**How YOLOv8 assigns locations without anchors.** Every cell of the stride 8 / 16 / 32 feature maps
predicts the distances from its centre to the four box sides (as a distribution, DFL) and class scores.
Positives are chosen by the Task-Aligned Assigner (TOOD): among the cells whose centre lies inside a
GT box, it keeps the top-10 by `score^0.5 · IoU^6`. A small object covers very few cells
(an 8×8 px object covers one stride-8 cell), so small and densely packed objects compete for the
same few locations. The two experiments target exactly this.

| Run | Change vs baseline | Why |
|---|---|---|
| `yolo_base` | YOLOv8s, input 1280 | baseline |
| `yolo_1600` | input **1600** | input resolution: ~10% of objects are < 16 px at 1280 (EDA) |
| `yolo_p2` | extra **P2 head** (stride 4), input 1280 | finer grid: more locations per small object |

Approximate time on a T4 with `FAST=True`: ~2 h for all three runs (~4.5 h without FAST).

## 0. Setup
Works on **Kaggle** (add the competition as input, GPU, Internet on), **Google Colab** (GPU T4,
Kaggle token as Colab secret `KAGGLE_API_TOKEN`, or `kaggle.json` in the root of Google Drive) and **a laptop** (Kaggle token in `<repo>/.kaggle_token`, git-ignored;
data and results go to `~/vdet_data` and `~/vdet_work`; without a GPU use it only with `SMOKE = True`).
On Colab all runs are saved to `MyDrive/vdet`, so after a disconnect just run the notebook again:
finished runs are skipped and interrupted runs continue from the last epoch.

In [ ]:
# ---- Settings ----
SMOKE = False   # True: tiny run on a few images, only to check that the notebook works end to end
FAST = True     # True: every 20th train frame and fewer epochs (~2x faster); False: full protocol
REPO = "https://github.com/LidaSokha03/AI_HW2.git"   # shared code is in src/ of this repository

In [ ]:
import os
import sys
import subprocess

if "google.colab" in sys.modules:  # Colab: keep runs and results on Google Drive
    from google.colab import drive
    drive.mount("/content/drive")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'ultralytics==8.4.207', 'pycocotools', 'kaggle>=2.0'], check=True)

# Shared code (src/vdet.py, src/tvdet.py): local copy if present, otherwise cloned from GitHub
CANDIDATES = ["src", "../src", ".", "notebooks"]
SRC = os.environ.get("VDET_SRC") or next((p for p in CANDIDATES if os.path.exists(f"{p}/vdet.py")), None)
if SRC is None:
    if os.path.exists("/tmp/AI_HW2"):
        subprocess.run(["git", "-C", "/tmp/AI_HW2", "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "/tmp/AI_HW2"], check=True)
    SRC = next(f"/tmp/AI_HW2/{p}" for p in ["src", "notebooks"] if os.path.exists(f"/tmp/AI_HW2/{p}/vdet.py"))
sys.path.insert(0, os.path.abspath(SRC))

import vdet

# Same frame subset for every model (see the protocol below). First run: ~5 min to prepare.
TRAIN_STEP = 20 if FAST else 10
S = vdet.setup(train_step=TRAIN_STEP, val_step=5, max_side=1600, smoke=SMOKE)
print({k: S[k] for k in ["platform", "work", "prep"]})

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

NB = "04_yolo"
GT = f"{S['prep']}/val_gt.json"
VAL = vdet.val_items(S)[:20] if SMOKE else vdet.val_items(S)
TEST = vdet.test_items(S)[:30] if SMOKE else vdet.test_items(S)
FP16 = vdet.fp16_args()

### Common protocol (identical in all model notebooks)
| Item | Value |
|---|---|
| Split | given train / val split, which is already **by video** (56 / 7 videos, no leakage) |
| Frames | every 20th train frame (≈1,200, `FAST=True`) or every 10th (≈2,400) and every 5th val frame (≈570): neighbouring frames are near-duplicates |
| Resolution | frames stored with longer side ≤ 1600 px; model input longer side **1280 px** unless the experiment changes it |
| Targets | `iscrowd=1` boxes (ignored regions, "others") are removed from training targets |
| Metric | COCO mAP@[.5:.95], AP50, AP_small… with `pycocotools` on val at **original resolution**, ignored regions as `iscrowd`, ≤ 500 detections per frame (same as the leaderboard) |
| Speed | ms per frame on the same GPU, batch 1, including image reading and NMS |
| Budget | ≈ 0.5 GPU-hour (T4) per run with `FAST=True`, ≈ 1 hour without |
| Seed | 42 |

### Training settings
SGD (lr 0.01), 12 epochs (20 without FAST), default Ultralytics augmentation (mosaic, HSV, flip,
scale ±50%), mosaic switched off for the last quarter of the epochs. Ultralytics accumulates gradients to a nominal batch of 64,
so runs with batch 4 and 8 get the same effective batch.

In [ ]:
EPOCHS = 1 if SMOKE else (12 if FAST else 20)
COMMON = dict(epochs=EPOCHS, optimizer="SGD", lr0=0.01, close_mosaic=max(1, EPOCHS // 4), max_det=500,
              cache=True, deterministic=True, fraction=0.05 if SMOKE else 1.0)
RUNS = {
    "yolo_base": dict(model_src="yolov8s.pt", imgsz=1280, batch=8),
    "yolo_1600": dict(model_src="yolov8s.pt", imgsz=1600, batch=4),
    "yolo_p2":   dict(model_src="yolov8s-p2.yaml", load_weights="yolov8s.pt", imgsz=1280, batch=4),
}
COLS = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR500", "ms_per_img", "FPS"]
results, val_preds = {}, {}


def run_yolo(name):
    """Train (or reuse) one run, evaluate it with the shared COCO evaluator and log the result."""
    cfg = RUNS[name]
    run_dir = vdet.ultra_train(S, name, cfg["model_src"], imgsz=cfg["imgsz"], batch=cfg["batch"],
                               load_weights=cfg.get("load_weights"), **COMMON)
    model = YOLO(f"{run_dir}/weights/best.pt")
    preds = vdet.ultra_predict(model, VAL, imgsz=cfg["imgsz"])
    metrics = vdet.coco_eval(GT, preds, per_class=True)
    metrics.update(vdet.measure_speed(
        lambda p: model.predict(p, imgsz=cfg["imgsz"], conf=0.001, max_det=500, verbose=False, **FP16),
        [it["path"] for it in VAL], n=10 if SMOKE else 100))
    vdet.log_result(S, NB, name, metrics, imgsz=cfg["imgsz"], epochs=EPOCHS, model=cfg["model_src"])
    results[name], val_preds[name] = metrics, preds
    return pd.DataFrame([metrics], index=[name])[COLS]

## 1. Baseline: YOLOv8s, input 1280

In [ ]:
run_yolo("yolo_base")

Early submission with the baseline, to check the file format and that we beat the competition baseline.

In [ ]:
model = YOLO(f"{S['work']}/runs/yolo_base/weights/best.pt")
vdet.write_submission(S, vdet.ultra_predict(model, TEST, imgsz=1280), "yolo_base")

## 2. Experiment 1: input resolution 1600
Same model, larger input. Expected: better AP_small, slower inference.

In [ ]:
run_yolo("yolo_1600")

## 3. Experiment 2: extra high-resolution head (P2, stride 4)
`yolov8s-p2.yaml` adds a fourth detection level at stride 4. All layers that exist in YOLOv8s are
initialised from the COCO weights; the new P2 layers start from scratch.

In [ ]:
run_yolo("yolo_p2")

## 4. Comparison

In [ ]:
table = vdet.results_table(results)
display(table[COLS])
vdet.plot_size_bars(table, "YOLOv8: AP by object size", save=f"{S['work']}/figs/yolo_ap_by_size.png")
print("AP per class:")
display(table[[f"AP_{c}" for c in vdet.CLASS_NAMES]].astype(float).round(3))

Training curves. Note: the mAP here is Ultralytics' own validation, which does **not** know about
ignored regions, so it is lower than our COCO evaluation. It is only used to compare curves.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name in RUNS:
    df = vdet.read_ultra_results(f"{S['work']}/runs/{name}")
    axes[0].plot(df["epoch"], df["metrics/mAP50-95(B)"], marker="o", label=name)
    axes[1].plot(df["epoch"], df["train/box_loss"], marker="o", label=name)
axes[0].set_title("val mAP50-95 (Ultralytics)"); axes[1].set_title("train box loss")
for ax in axes:
    ax.set_xlabel("epoch"); ax.legend()
plt.tight_layout(); plt.savefig(f"{S['work']}/figs/yolo_curves.png", dpi=110); plt.show()

### Comparison with the anchor-based detectors, by object size
Read from `results/02_faster_rcnn.csv` and `results/03_retinanet.csv` (this work dir, Kaggle inputs or
the repository's `results/` folder).

In [ ]:
allres = vdet.load_results(S)
size_cols = ["mAP", "AP50", "AP_small", "AP_medium", "AP_large", "ms_per_img"]
if len(allres) and allres.notebook.isin(["02_faster_rcnn", "03_retinanet"]).any():
    others = allres[allres.notebook.isin(["02_faster_rcnn", "03_retinanet"])].set_index("run")
    cmp = pd.concat([others, table])[size_cols].astype(float).round(4)
    display(cmp)
    vdet.plot_size_bars(cmp, "Anchor-based vs anchor-free: AP by object size",
                        save=f"{S['work']}/figs/anchor_based_vs_free.png")
else:
    print("No Faster R-CNN / RetinaNet results found yet (run 02 and 03 or put their results/*.csv into the repository).")

## 5. Errors of the best run (small and dense objects)

In [ ]:
best = table["mAP"].astype(float).idxmax()
print("best run:", best)
miss = vdet.error_breakdown(S, val_preds[best])
ids = vdet.worst_frames(S, val_preds[best], k=2)
vdet.plot_errors(S, val_preds[best], ids, save=f"{S['work']}/figs/yolo_errors.png")

## 6. Submission with the best run

In [ ]:
if best != "yolo_base":
    model = YOLO(f"{S['work']}/runs/{best}/weights/best.pt")
    vdet.write_submission(S, vdet.ultra_predict(model, TEST, imgsz=RUNS[best]["imgsz"]), best)